In [1]:
# pip install smolagents requests
# pip install 'smolagents[transformers]'

from smolagents import CodeAgent, tool, TransformersModel, ToolCallingAgent
import requests

from smolagents import (
    CodeAgent,
    ToolCallingAgent,
    TransformersModel,
    PythonInterpreterTool,
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    tool,
)
from smolagents.memory import ActionStep, FinalAnswerStep, SystemPromptStep, TaskStep

In [2]:
model = TransformersModel(model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
OPENWEATHER_API_KEY = "9442177234af4594189a800863a98fd3"

@tool
def weather_forecast(city: str) -> dict:
    """
    Get current weather for a city
    
    Args:
        city: Name of the city to fetch the weather for

    Returns a dictionary with:
        - city (str): Name of the city
        - forecasts (list[dict]): A list of 5 objects, each containing:
            - date (str): The date of the forecast (YYYY-MM-DD)
            - temperature (float): Predicted midday temperature in Celsius
            - description (str): Weather condition description
    """
    
    #url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    res = requests.get(url).json()

    daily_snapshots = res["list"][:40:8] 

    forecast_data = []
    for item in daily_snapshots:
        forecast_data.append({
            "date": item["dt_txt"],
            "temperature": item["main"]["temp"],
            "description": item["weather"][0]["description"]
        })

    return {
        "city": city,
        "forecasts": forecast_data
    }

tc_agent = ToolCallingAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool(), weather_forecast],
    model=model,
    max_steps=4,
    #instructions
    )

TASK = "What will the weather be like in Rome for the next 3 days?"

print(tc_agent.system_prompt)

result_tc = tc_agent.run(TASK)
print("ToolCallingAgent answer:", result_tc)

You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you can.
To do so, you have been given access to some tools.

The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".
This Action/Observation can repeat N times, you should take several steps when needed.

You can use the result of the previous action as input for the next action.
The observation will always be a string: it can represent a file, like "image_1.jpg".
Then you can use it as input for the next action. You can do it for instance as follows:

Observation: "image_1.jpg"

Action:
{
  "name": "image_transformer",
  "arguments": {"image": "image_1.jpg"}
}

To provide the final answer to the task, use an action blob with "name": "final_answer" tool. It is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:
Action:
{
  "name": "final_answer"

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What will the weather be like in Rome for the next 3 days?                                                      │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'weather_forecast' with arguments: {'city': 'Rome'}                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'city': 'Rome', 'forecasts': |{'date': '2026-04-30 15:00:00', 'temperature': 15.79, 'description': 
'scattered clouds'}, {'date': '2026-05-01 15:00:00', 'temperature': 14.23, 'description': 'overcast clouds'}, 
{'date': '2026-05-02 15:00:00', 'temperature': 14.74, 'description': 'overcast clouds'}, {'date': '2026-05-03 
15:00:00', 'temperature': 17.41, 'description': 'clear sky'}, {'date': '2026-05-04 15:00:00', 'temperature': 20.35,
'description': 'clear sky'}]}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The weather in Rome for the next 3 days will be        │
│ scattered clouds on April 30th, overcast clouds on May 1st, overcast clouds on May 2nd, clear sky on May 3rd,   │
│ and clear sky on May 4th.'}                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The weather in Rome for the next 3 days will be scattered clouds on April 30th, overcast clouds on 
May 1st, overcast clouds on May 2nd, clear sky on May 3rd, and clear sky on May 4th.

Final answer: The weather in Rome for the next 3 days will be scattered clouds on April 30th, overcast clouds on 
May 1st, overcast clouds on May 2nd, clear sky on May 3rd, and clear sky on May 4th.

[Step 2: Duration 48.15 seconds| Input tokens: 3,357 | Output tokens: 114]

ToolCallingAgent answer: The weather in Rome for the next 3 days will be scattered clouds on April 30th, overcast clouds on May 1st, overcast clouds on May 2nd, clear sky on May 3rd, and clear sky on May 4th.


In [5]:
import requests

city="Rome"
#url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
res = requests.get(url).json()

print(res)
print(len(res["list"]))

for item in res["list"][:10]:
    print(item["dt_txt"], item["main"]["temp"])

daily_snapshots = res["list"][:40:8] 
print(len(daily_snapshots))
forecast_data = []
for item in daily_snapshots:
    forecast_data.append({
        "date": item["dt_txt"],
        "temperature": item["main"]["temp"],
        "description": item["weather"][0]["description"]
    })

for f in forecast_data:
    print(f)

{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1777550400, 'main': {'temp': 14.61, 'feels_like': 14.63, 'temp_min': 14.61, 'temp_max': 15.07, 'pressure': 1012, 'sea_level': 1012, 'grnd_level': 986, 'humidity': 96, 'temp_kf': -0.46}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10d'}], 'clouds': {'all': 40}, 'wind': {'speed': 3.39, 'deg': 340, 'gust': 9.35}, 'visibility': 10000, 'pop': 1, 'rain': {'3h': 2.03}, 'sys': {'pod': 'd'}, 'dt_txt': '2026-04-30 12:00:00'}, {'dt': 1777561200, 'main': {'temp': 15.87, 'feels_like': 15.78, 'temp_min': 15.87, 'temp_max': 18.39, 'pressure': 1013, 'sea_level': 1013, 'grnd_level': 987, 'humidity': 87, 'temp_kf': -2.52}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'clouds': {'all': 60}, 'wind': {'speed': 5.21, 'deg': 360, 'gust': 8.46}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'd'}, 'dt_txt': '2026-04-30 15:00:00'}, {'dt': 1777572000, 'main': {'temp': 20.94, 'fe